
Trajectory optimisation prototype for a synthetic air-traffic management case study.

This code selects one feasible trajectory profile for each flight while accounting
for operational modification cost, weather exposure, and pairwise separation conflicts.

The solution method is a conflict-driven restricted MILP:
1. Build one feasible seed profile per flight.
2. Solve a restricted master MILP.
3. If selected profiles still conflict, identify flights to repair.
4. Generate targeted alternative profiles.
5. Re-solve until a conflict-free solution is found or no useful repair remains.

The dataset used here is synthetic and intended for demonstration and validation, not for operational deployment.


In [1]:
pip install pulp

Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import itertools
import math
from collections import Counter, defaultdict
from dataclasses import dataclass
from typing import Dict, Iterable, List, Sequence, Tuple

import pulp


# ============================================================
# 1. Configuration
# ============================================================

@dataclass(frozen=True)
class ModelConfig:
    # Conflict-driven dynamic profile generation
    initial_profiles_per_flight: int = 1
    profiles_added_per_flight: int = 4
    max_profiles_per_flight: int = 35
    max_dynamic_iterations: int = 20
    conflict_slack_penalty: float = 10000.0
    candidate_conflict_penalty: float = 1000.0
    # Generate upward alternatives first. The objective still decides; this is
    # only a deterministic climb-first ordering for otherwise equal candidates.
    local_fl_offsets: Tuple[int, ...] = (10, 20, -10, -20)
    local_time_offsets: Tuple[float, ...] = (-10.0, -5.0, 5.0, 10.0)
    local_mach_offsets: Tuple[float, ...] = (-0.02, 0.02)

    # Weather
    hard_weather_limit: float = 250.0
    weather_cost_weight: float = 0.02

    # Conflict model
    # FL values are written as 330, 340, ...
    # 20 means that profiles at the same or adjacent flight levels conflict
    # whenever they share a segment too closely in time.
    # Set this to 10.0 to recover "same FL only".
    vertical_conflict_band_fl: float = 10.0

    # Request costs. Flight-level deviations are asymmetric:
    # climbing is preferred to descending. Costs are per 10 FL units.
    request_climb_step_cost: float = 2.0
    request_descent_step_cost: float = 6.0
    request_mach_step_cost: float = 1.0
    request_time_minute_cost: float = 0.2
    request_track_change_cost: float = 5.0
    request_direct_route_cost: float = 2.0

    # Existing-flight costs. Descending an existing flight is penalised
    # more heavily than climbing it.
    existing_disruption_cost: float = 8.0
    existing_climb_step_cost: float = 3.0
    existing_descent_step_cost: float = 9.0
    existing_mach_step_cost: float = 1.0
    existing_time_minute_cost: float = 0.2
    existing_track_change_cost: float = 6.0
    existing_direct_route_cost: float = 2.0

    # Output
    export_model: bool = False
    solver_messages: bool = True


CONFIG = ModelConfig()


# ============================================================
# 2. Data classes
# ============================================================

@dataclass(frozen=True)
class Track:
    track_id: str
    waypoints: Tuple[str, ...]
    allowed_fls: Tuple[int, ...]
    segment_minutes: Tuple[float, ...]

    def __post_init__(self) -> None:
        if len(self.segment_minutes) != len(self.waypoints) - 1:
            raise ValueError(
                f"Track {self.track_id}: len(segment_minutes) must equal "
                "len(waypoints) - 1."
            )


@dataclass(frozen=True)
class Flight:
    flight_id: str
    flight_type: str
    primary_track: str
    entry_time: float
    preferred_fl: int
    preferred_mach: float
    separation_standard: str
    min_fl: int
    max_fl: int
    feasible_tracks: Tuple[str, ...]

    def __post_init__(self) -> None:
        if self.flight_type not in {"existing", "request"}:
            raise ValueError(
                f"{self.flight_id}: flight_type must be existing or request."
            )
        if self.min_fl > self.max_fl:
            raise ValueError(f"{self.flight_id}: min_fl exceeds max_fl.")


@dataclass(frozen=True)
class WeatherCell:
    track_id: str
    start_time: float
    end_time: float
    min_fl: int
    max_fl: int
    severity: float


@dataclass(frozen=True)
class Segment:
    from_wp: str
    to_wp: str
    start_time: float
    end_time: float


@dataclass(frozen=True)
class Profile:
    flight_id: str
    profile_id: str
    track_id: str
    flight_level: int
    mach: float
    entry_time: float
    time_shift: float
    route_variant: str
    segments: Tuple[Segment, ...]
    weather_exposure: float
    total_cost: float

    @property
    def uses_direct_route(self) -> bool:
        return self.route_variant == "direct"


# ============================================================
# 3. Synthetic proof-of-concept data
# ============================================================

TRACKS: Dict[str, Track] = {
    "A": Track(
        "A",
        ("SUNOT", "57N020W", "58N030W", "58N040W", "58N050W", "CUDDY"),
        (340, 350, 360, 370, 380, 390),
        (42, 43, 44, 45, 42),
    ),
    "B": Track(
        "B",
        ("DOGAL", "56N020W", "57N030W", "57N040W", "57N050W", "HOIST"),
        (340, 350, 360, 370, 380, 390),
        (41, 42, 43, 44, 41),
    ),
    "C": Track(
        "C",
        ("MALOT", "55N020W", "56N030W", "56N040W", "56N050W", "JANJO"),
        (330, 340, 350, 360, 370, 380),
        (40, 41, 42, 43, 40),
    ),
    "D": Track(
        "D",
        ("LIMRI", "54N020W", "55N030W", "55N040W", "55N050W", "LOACH"),
        (330, 340, 350, 360, 370, 380),
        (39, 40, 41, 42, 39),
    ),
}


def make_flight(
    flight_id: str,
    flight_type: str,
    primary_track: str,
    entry_time: float,
    preferred_fl: int,
    preferred_mach: float,
    separation_standard: str,
    min_fl: int,
    max_fl: int,
    feasible_tracks: Sequence[str],
) -> Flight:
    return Flight(
        flight_id=flight_id,
        flight_type=flight_type,
        primary_track=primary_track,
        entry_time=float(entry_time),
        preferred_fl=preferred_fl,
        preferred_mach=preferred_mach,
        separation_standard=separation_standard,
        min_fl=min_fl,
        max_fl=max_fl,
        feasible_tracks=tuple(feasible_tracks),
    )


FLIGHTS: Dict[str, Flight] = {
    "EXG001": make_flight("EXG001", "existing", "A", 600, 370, .84, "PBCS_ONLY", 350, 390, ("A", "B")),
    "EXG002": make_flight("EXG002", "existing", "A", 608, 350, .82, "ASEPS", 340, 380, ("A", "B")),
    "EXG003": make_flight("EXG003", "existing", "B", 616, 370, .83, "PBCS_ONLY", 350, 390, ("A", "B", "C")),
    "EXG004": make_flight("EXG004", "existing", "B", 625, 360, .81, "ASEPS", 340, 380, ("B", "C")),
    "EXG005": make_flight("EXG005", "existing", "C", 633, 350, .82, "SPACE_ADSB_ONLY", 330, 380, ("B", "C", "D")),
    "EXG006": make_flight("EXG006", "existing", "C", 642, 370, .84, "PBCS_ONLY", 350, 390, ("C", "D")),
    "EXG007": make_flight("EXG007", "existing", "D", 650, 340, .80, "ASEPS", 330, 370, ("C", "D")),
    "EXG008": make_flight("EXG008", "existing", "D", 659, 360, .82, "PBCS_ONLY", 340, 380, ("C", "D")),
    "EXG009": make_flight("EXG009", "existing", "A", 668, 380, .85, "PBCS_ONLY", 360, 390, ("A", "B")),
    "EXG010": make_flight("EXG010", "existing", "B", 677, 340, .80, "ASEPS", 330, 370, ("A", "B", "C")),
    "REQ001": make_flight("REQ001", "request", "A", 604, 370, .84, "PBCS_ONLY", 350, 390, ("A", "B")),
    "REQ002": make_flight("REQ002", "request", "A", 611, 350, .82, "ASEPS", 340, 380, ("A", "B")),
    "REQ003": make_flight("REQ003", "request", "B", 619, 370, .83, "PBCS_ONLY", 350, 390, ("A", "B", "C")),
    "REQ004": make_flight("REQ004", "request", "B", 628, 360, .81, "ASEPS", 340, 380, ("B", "C")),
    "REQ005": make_flight("REQ005", "request", "C", 636, 350, .82, "SPACE_ADSB_ONLY", 330, 380, ("B", "C", "D")),
    "REQ006": make_flight("REQ006", "request", "C", 645, 370, .84, "PBCS_ONLY", 350, 390, ("C", "D")),
    "REQ007": make_flight("REQ007", "request", "D", 653, 340, .80, "ASEPS", 330, 370, ("C", "D")),
    "REQ008": make_flight("REQ008", "request", "D", 662, 360, .82, "PBCS_ONLY", 340, 380, ("C", "D")),
    "REQ009": make_flight("REQ009", "request", "A", 671, 380, .85, "PBCS_ONLY", 360, 390, ("A", "B")),
    "REQ010": make_flight("REQ010", "request", "B", 680, 340, .80, "ASEPS", 330, 370, ("A", "B", "C")),
}


WEATHER: Tuple[WeatherCell, ...] = (
    WeatherCell("A", 590, 670, 360, 390, 3.0),
    WeatherCell("B", 610, 700, 350, 380, 2.0),
    WeatherCell("C", 625, 710, 340, 370, 4.0),
    WeatherCell("D", 640, 720, 330, 360, 2.5),
)

SEPARATION: Dict[str, float] = {
    "ASEPS": 3.0,
    "PBCS_ONLY": 5.0,
    "SPACE_ADSB_ONLY": 10.0,
}

ENTRY_SHIFTS = (-10.0, -5.0, 0.0, 5.0, 10.0)
MACH_SHIFTS = (-0.02, 0.0, 0.02)
ROUTE_VARIANTS = ("nominal", "direct")


# ============================================================
# 4. Profile construction
# ============================================================

def build_segments(
    track: Track,
    entry_time: float,
    mach: float,
    route_variant: str,
) -> Tuple[Segment, ...]:
    waypoints = list(track.waypoints)
    durations = list(track.segment_minutes)

    if route_variant == "direct":
        if len(waypoints) < 6 or len(durations) < 4:
            raise ValueError(
                f"Track {track.track_id} is too short for direct routing."
            )

        # Remove one middle waypoint and merge its neighboring segments.
        waypoints.pop(3)
        durations[2:4] = [0.92 * (durations[2] + durations[3])]

    speed_factor = 0.82 / mach
    current_time = entry_time
    segments: List[Segment] = []

    for index, nominal_duration in enumerate(durations):
        end_time = current_time + nominal_duration * speed_factor
        segments.append(
            Segment(
                from_wp=waypoints[index],
                to_wp=waypoints[index + 1],
                start_time=current_time,
                end_time=end_time,
            )
        )
        current_time = end_time

    return tuple(segments)


def weather_exposure(
    track_id: str,
    flight_level: int,
    segments: Sequence[Segment],
) -> float:
    exposure = 0.0

    for cell in WEATHER:
        if cell.track_id != track_id:
            continue
        if not (cell.min_fl <= flight_level <= cell.max_fl):
            continue

        for segment in segments:
            overlap = max(
                0.0,
                min(segment.end_time, cell.end_time)
                - max(segment.start_time, cell.start_time),
            )
            exposure += overlap * cell.severity

    return exposure


def profile_cost(
    flight: Flight,
    track_id: str,
    flight_level: int,
    mach: float,
    entry_time: float,
    route_variant: str,
    weather_value: float,
    config: ModelConfig,
) -> float:
    fl_difference = flight_level - flight.preferred_fl
    climb_steps = max(0.0, fl_difference) / 10.0
    descent_steps = max(0.0, -fl_difference) / 10.0
    mach_steps = abs(mach - flight.preferred_mach) / 0.01
    time_deviation = abs(entry_time - flight.entry_time)
    track_changed = track_id != flight.primary_track
    route_changed = route_variant != "nominal"

    if flight.flight_type == "existing":
        any_change = (
            climb_steps > 0
            or descent_steps > 0
            or mach_steps > 0
            or time_deviation > 0
            or track_changed
            or route_changed
        )
        cost = (
            (config.existing_disruption_cost if any_change else 0.0)
            + config.existing_climb_step_cost * climb_steps
            + config.existing_descent_step_cost * descent_steps
            + config.existing_mach_step_cost * mach_steps
            + config.existing_time_minute_cost * time_deviation
            + config.existing_track_change_cost * float(track_changed)
            + config.existing_direct_route_cost * float(route_changed)
            + config.weather_cost_weight * weather_value
        )
    else:
        cost = (
            config.request_climb_step_cost * climb_steps
            + config.request_descent_step_cost * descent_steps
            + config.request_mach_step_cost * mach_steps
            + config.request_time_minute_cost * time_deviation
            + config.request_track_change_cost * float(track_changed)
            + config.request_direct_route_cost * float(route_changed)
            + config.weather_cost_weight * weather_value
        )

    return round(cost, 3)


# ============================================================
# 5. Conflict-driven dynamic profile generation
# ============================================================

@dataclass(frozen=True)
class ConflictInfo:
    profile_a_id: str
    profile_b_id: str
    flight_a_id: str
    flight_b_id: str
    segment_a: Tuple[str, str]
    segment_b: Tuple[str, str]
    gap: float
    required_separation: float
    vertical_difference: float

    @property
    def separation_deficit(self) -> float:
        return max(0.0, self.required_separation - self.gap)


@dataclass
class RestrictedSolution:
    status: str
    objective: float | None
    selected_by_flight: Dict[str, Profile]
    active_conflicts: List[ConflictInfo]
    conflict_constraints: int
    paid_conflict_slacks: int


def interval_gap(
    a_start: float,
    a_end: float,
    b_start: float,
    b_end: float,
) -> float:
    if a_end < b_start:
        return b_start - a_end
    if b_end < a_start:
        return a_start - b_end
    return 0.0


def same_segment(a: Segment, b: Segment) -> bool:
    return (
        (a.from_wp, a.to_wp) == (b.from_wp, b.to_wp)
        or (a.from_wp, a.to_wp) == (b.to_wp, b.from_wp)
    )


def diagnose_conflict(
    profile_a: Profile,
    profile_b: Profile,
    config: ModelConfig,
) -> ConflictInfo | None:
    if profile_a.flight_id == profile_b.flight_id:
        return None

    vertical_difference = abs(
        profile_a.flight_level - profile_b.flight_level
    )
    if vertical_difference >= config.vertical_conflict_band_fl:
        return None

    flight_a = FLIGHTS[profile_a.flight_id]
    flight_b = FLIGHTS[profile_b.flight_id]
    required = max(
        SEPARATION[flight_a.separation_standard],
        SEPARATION[flight_b.separation_standard],
    )

    worst: ConflictInfo | None = None
    for segment_a in profile_a.segments:
        for segment_b in profile_b.segments:
            if not same_segment(segment_a, segment_b):
                continue
            gap = interval_gap(
                segment_a.start_time,
                segment_a.end_time,
                segment_b.start_time,
                segment_b.end_time,
            )
            if gap >= required:
                continue
            info = ConflictInfo(
                profile_a_id=profile_a.profile_id,
                profile_b_id=profile_b.profile_id,
                flight_a_id=profile_a.flight_id,
                flight_b_id=profile_b.flight_id,
                segment_a=(segment_a.from_wp, segment_a.to_wp),
                segment_b=(segment_b.from_wp, segment_b.to_wp),
                gap=gap,
                required_separation=required,
                vertical_difference=vertical_difference,
            )
            if worst is None or info.separation_deficit > worst.separation_deficit:
                worst = info
    return worst


def conflict(
    profile_a: Profile,
    profile_b: Profile,
    config: ModelConfig,
) -> bool:
    return diagnose_conflict(profile_a, profile_b, config) is not None


def profile_signature(
    track_id: str,
    flight_level: int,
    mach: float,
    entry_time: float,
    route_variant: str,
) -> Tuple[str, int, int, int, str]:
    return (
        track_id,
        int(flight_level),
        int(round(mach * 100)),
        int(round(entry_time * 10)),
        route_variant,
    )



def signed_token(value: float) -> str:
    """Return a stable, identifier-safe token for a signed numeric value."""
    rounded = int(round(value))
    return f"m{abs(rounded)}" if rounded < 0 else f"p{rounded}"

def build_profile(
    flight: Flight,
    track_id: str,
    flight_level: int,
    mach: float,
    entry_time: float,
    route_variant: str,
    config: ModelConfig,
) -> Profile | None:
    if track_id not in flight.feasible_tracks or track_id not in TRACKS:
        return None
    track = TRACKS[track_id]
    if flight_level not in track.allowed_fls:
        return None
    if not flight.min_fl <= flight_level <= flight.max_fl:
        return None
    mach = round(mach, 2)
    if not 0.78 <= mach <= 0.87:
        return None
    if route_variant not in ROUTE_VARIANTS:
        return None

    segments = build_segments(track, entry_time, mach, route_variant)
    wx = weather_exposure(track_id, flight_level, segments)
    if wx > config.hard_weather_limit:
        return None

    time_shift = entry_time - flight.entry_time
    pid = (
        f"{flight.flight_id}_T{track_id}_FL{flight_level}"
        f"_M{int(round(mach * 100)):02d}"
        f"_DT{signed_token(time_shift)}_R{route_variant}"
    )
    return Profile(
        flight_id=flight.flight_id,
        profile_id=pid,
        track_id=track_id,
        flight_level=flight_level,
        mach=mach,
        entry_time=entry_time,
        time_shift=time_shift,
        route_variant=route_variant,
        segments=segments,
        weather_exposure=wx,
        total_cost=profile_cost(
            flight,
            track_id,
            flight_level,
            mach,
            entry_time,
            route_variant,
            wx,
            config,
        ),
    )


def generate_initial_profiles(
    flight: Flight,
    config: ModelConfig,
) -> List[Profile]:
    """Return exactly one feasible seed profile.

    The nominal profile is tried first. Only when it violates a hard
    feasibility condition (for example, the weather exposure limit) do we
    search a small deterministic fallback list. No Cartesian-product
    enumeration is performed.
    """
    nominal = build_profile(
        flight,
        flight.primary_track,
        flight.preferred_fl,
        flight.preferred_mach,
        flight.entry_time,
        "nominal",
        config,
    )
    if nominal is not None:
        return [nominal]

    fallback_specs: List[Tuple[str, int, float, float, str]] = []

    # First try nearby flight levels on the primary track.
    for dfl in (-10, 10, -20, 20):
        fallback_specs.append((
            flight.primary_track,
            flight.preferred_fl + dfl,
            flight.preferred_mach,
            flight.entry_time,
            "nominal",
        ))

    # Then nearby times.
    for dt in (-5.0, 5.0, -10.0, 10.0):
        fallback_specs.append((
            flight.primary_track,
            flight.preferred_fl,
            flight.preferred_mach,
            flight.entry_time + dt,
            "nominal",
        ))

    # Then feasible alternate tracks, followed by small Mach/route changes.
    for track_id in flight.feasible_tracks:
        if track_id != flight.primary_track:
            fallback_specs.append((
                track_id,
                flight.preferred_fl,
                flight.preferred_mach,
                flight.entry_time,
                "nominal",
            ))
    for dm in (-0.02, 0.02):
        fallback_specs.append((
            flight.primary_track,
            flight.preferred_fl,
            round(flight.preferred_mach + dm, 2),
            flight.entry_time,
            "nominal",
        ))
    fallback_specs.append((
        flight.primary_track,
        flight.preferred_fl,
        flight.preferred_mach,
        flight.entry_time,
        "direct",
    ))

    feasible: List[Profile] = []
    seen = set()
    for spec in fallback_specs:
        signature = profile_signature(*spec)
        if signature in seen:
            continue
        seen.add(signature)
        candidate = build_profile(flight, *spec, config)
        if candidate is not None:
            feasible.append(candidate)

    if not feasible:
        raise ValueError(f"{flight.flight_id}: no feasible seed profile.")

    feasible.sort(key=lambda p: (p.total_cost, p.profile_id))
    return [feasible[0]]


def build_nominal_profiles(
    config: ModelConfig,
) -> Tuple[Dict[str, Profile], List[str]]:
    """Build nominal profiles without generating any alternatives.

    Returns the feasible nominal profiles and the IDs whose nominal profile
    violates a hard feasibility condition.
    """
    nominal_by_flight: Dict[str, Profile] = {}
    infeasible_nominal: List[str] = []

    for fid, flight in FLIGHTS.items():
        profile = build_profile(
            flight,
            flight.primary_track,
            flight.preferred_fl,
            flight.preferred_mach,
            flight.entry_time,
            "nominal",
            config,
        )
        if profile is None:
            infeasible_nominal.append(fid)
        else:
            nominal_by_flight[fid] = profile

    return nominal_by_flight, infeasible_nominal

def selected_conflicts(
    selected_by_flight: Dict[str, Profile],
    config: ModelConfig,
) -> List[ConflictInfo]:
    infos: List[ConflictInfo] = []
    selected = list(selected_by_flight.values())
    for a, b in itertools.combinations(selected, 2):
        info = diagnose_conflict(a, b, config)
        if info is not None:
            infos.append(info)
    return infos


def choose_solver(msg: bool = True):
    available = set(pulp.listSolvers(onlyAvailable=True))
    print("AVAILABLE SOLVERS:", sorted(available))
    if "HiGHS" in available:
        return pulp.HiGHS(msg=msg, timeLimit=600, gapRel=0.01)
    if "PULP_CBC_CMD" in available:
        return pulp.PULP_CBC_CMD(
            msg=msg,
            timeLimit=600,
            gapRel=0.01,
            threads=4,
        )
    raise RuntimeError("No supported MILP solver is available.")


def solve_restricted_master(
    profiles: Dict[str, List[Profile]],
    config: ModelConfig,
) -> RestrictedSolution:
    all_profiles = [p for pool in profiles.values() for p in pool]
    by_id = {p.profile_id: p for p in all_profiles}
    if len(by_id) != len(all_profiles):
        raise ValueError("Duplicate profile IDs in dynamic pool.")

    model = pulp.LpProblem("Conflict_Driven_Dynamic_Master", pulp.LpMinimize)
    x = {
        p.profile_id: pulp.LpVariable(f"x_{p.profile_id}", cat="Binary")
        for p in all_profiles
    }

    conflict_pairs: List[Tuple[Profile, Profile, pulp.LpVariable]] = []
    conflict_count = 0
    flight_ids = list(FLIGHTS)
    for i, fid_a in enumerate(flight_ids):
        for fid_b in flight_ids[i + 1:]:
            for a in profiles[fid_a]:
                for b in profiles[fid_b]:
                    if not conflict(a, b, config):
                        continue
                    z = pulp.LpVariable(
                        f"z_conflict_{conflict_count:07d}", cat="Binary"
                    )
                    model += x[a.profile_id] + x[b.profile_id] <= 1 + z
                    conflict_pairs.append((a, b, z))
                    conflict_count += 1

    model += (
        pulp.lpSum(p.total_cost * x[p.profile_id] for p in all_profiles)
        + config.conflict_slack_penalty
        * pulp.lpSum(z for _, _, z in conflict_pairs)
    )

    # Every flight must be assigned exactly one profile. Rejection is forbidden.
    for fid, pool in profiles.items():
        model += pulp.lpSum(x[p.profile_id] for p in pool) == 1

    status_code = model.solve(choose_solver(config.solver_messages))
    status = pulp.LpStatus[status_code]
    if status not in {"Optimal", "Feasible"}:
        return RestrictedSolution(
            status=status,
            objective=None,
            selected_by_flight={},
            active_conflicts=[],
            conflict_constraints=conflict_count,
            paid_conflict_slacks=0,
        )

    selected_by_flight: Dict[str, Profile] = {}
    for fid, pool in profiles.items():
        chosen = [
            p for p in pool
            if (pulp.value(x[p.profile_id]) or 0.0) > 0.5
        ]
        if len(chosen) != 1:
            raise RuntimeError(
                f"{fid}: expected exactly one selected profile, got {len(chosen)}"
            )
        selected_by_flight[fid] = chosen[0]

    paid = sum(
        1 for _, _, z in conflict_pairs
        if (pulp.value(z) or 0.0) > 0.5
    )
    return RestrictedSolution(
        status=status,
        objective=float(pulp.value(model.objective)),
        selected_by_flight=selected_by_flight,
        active_conflicts=selected_conflicts(selected_by_flight, config),
        conflict_constraints=conflict_count,
        paid_conflict_slacks=paid,
    )

def candidate_specs_for_repair(
    flight: Flight,
    current: Profile,
    conflict_infos: Sequence[ConflictInfo],
    config: ModelConfig,
) -> List[Tuple[str, int, float, float, str, str]]:
    """Return (track, FL, Mach, entry, route, operator) specifications."""
    specs: List[Tuple[str, int, float, float, str, str]] = []

    # Temporal repair: move entry time by enough to cover the largest deficit,
    # rounded up to the local five-minute grid.
    max_deficit = max((c.separation_deficit for c in conflict_infos), default=0.0)
    adaptive_shift = max(5.0, 5.0 * math.ceil(max_deficit / 5.0))
    temporal_offsets = set(config.local_time_offsets) | {-adaptive_shift, adaptive_shift}
    for dt in sorted(temporal_offsets):
        specs.append((current.track_id, current.flight_level, current.mach,
                      current.entry_time + dt, current.route_variant, "time"))

    # Vertical repair.
    for dfl in config.local_fl_offsets:
        specs.append((current.track_id, current.flight_level + dfl, current.mach,
                      current.entry_time, current.route_variant, "flight_level"))

    # Route/track repair. Track changes can remove all common segments.
    for track_id in flight.feasible_tracks:
        if track_id != current.track_id:
            specs.append((track_id, current.flight_level, current.mach,
                          current.entry_time, "nominal", "track"))

    # Direct routing can remove a middle segment.
    if current.route_variant != "direct":
        specs.append((current.track_id, current.flight_level, current.mach,
                      current.entry_time, "direct", "route"))

    # Mach is a residual fine-tuning operator.
    for dm in config.local_mach_offsets:
        specs.append((current.track_id, current.flight_level,
                      round(current.mach + dm, 2), current.entry_time,
                      current.route_variant, "mach"))
    return specs


def generate_targeted_profiles(
    flight: Flight,
    current: Profile,
    conflict_infos: Sequence[ConflictInfo],
    selected_by_flight: Dict[str, Profile],
    existing_pool: Sequence[Profile],
    config: ModelConfig,
) -> List[Profile]:
    existing_signatures = {
        profile_signature(p.track_id, p.flight_level, p.mach,
                          p.entry_time, p.route_variant)
        for p in existing_pool
    }
    opponents = [
        p for fid, p in selected_by_flight.items()
        if fid != flight.flight_id
    ]

    ranked: List[Tuple[float, int, float, str, Profile]] = []
    for track_id, fl, mach, entry, route, operator in candidate_specs_for_repair(
        flight, current, conflict_infos, config
    ):
        signature = profile_signature(track_id, fl, mach, entry, route)
        if signature in existing_signatures:
            continue
        candidate = build_profile(
            flight, track_id, fl, mach, entry, route, config
        )
        if candidate is None:
            continue
        remaining = sum(conflict(candidate, other, config) for other in opponents)
        score = candidate.total_cost + config.candidate_conflict_penalty * remaining
        # Prefer candidates with fewer conflicts, then lower score/cost.
        ranked.append((score, remaining, candidate.total_cost, operator, candidate))

    ranked.sort(key=lambda item: (item[1], item[0], item[2], item[4].profile_id))
    room = max(0, config.max_profiles_per_flight - len(existing_pool))
    limit = min(config.profiles_added_per_flight, room)
    return [item[4] for item in ranked[:limit]]



def choose_repair_flights(
    active_conflicts: Sequence[ConflictInfo],
    selected_by_flight: Dict[str, Profile],
) -> List[str]:
    """Choose a small set of flights whose profile pools should be expanded.

    This is a greedy weighted vertex-cover heuristic on the active-conflict
    graph. A request is preferred over an existing flight, and among flights
    of the same type we prefer the one covering more still-uncovered conflicts.
    Only the selected endpoint(s) receive new profiles.
    """
    uncovered = list(active_conflicts)
    chosen: List[str] = []

    while uncovered:
        degree: Dict[str, int] = defaultdict(int)
        for info in uncovered:
            degree[info.flight_a_id] += 1
            degree[info.flight_b_id] += 1

        candidates = set(degree)

        def rank(fid: str) -> Tuple[int, int, float, str]:
            flight = FLIGHTS[fid]
            # Requests are cheaper/more flexible to modify than existing flights.
            type_priority = 0 if flight.flight_type == "request" else 1
            # Higher uncovered-conflict degree is better, hence the minus sign.
            current_cost = selected_by_flight[fid].total_cost
            return (type_priority, -degree[fid], current_cost, fid)

        selected_fid = min(candidates, key=rank)
        chosen.append(selected_fid)
        uncovered = [
            info for info in uncovered
            if selected_fid not in {info.flight_a_id, info.flight_b_id}
        ]

    return chosen

def describe_changes(flight: Flight, profile: Profile) -> str:
    changes: List[str] = []
    if profile.track_id != flight.primary_track:
        changes.append(f"track {flight.primary_track}->{profile.track_id}")
    if profile.flight_level != flight.preferred_fl:
        changes.append(f"FL {flight.preferred_fl}->{profile.flight_level}")
    if not math.isclose(profile.mach, flight.preferred_mach, abs_tol=1e-9):
        changes.append(f"Mach {flight.preferred_mach:.2f}->{profile.mach:.2f}")
    if not math.isclose(profile.entry_time, flight.entry_time, abs_tol=1e-9):
        changes.append(f"entry {flight.entry_time:.0f}->{profile.entry_time:.0f}")
    if profile.route_variant != "nominal":
        changes.append(f"route={profile.route_variant}")
    return ", ".join(changes) if changes else "unchanged"


def print_final_solution(solution: RestrictedSolution) -> None:
    print("\n" + "=" * 120)
    print("FINAL STATUS:", solution.status)
    print(
        "FINAL OBJECTIVE:",
        None if solution.objective is None else round(solution.objective, 3),
    )
    print("ACTIVE CONFLICTS:", len(solution.active_conflicts))
    print("-" * 120)
    for fid in FLIGHTS:
        p = solution.selected_by_flight.get(fid)
        if p is None:
            continue
        flight = FLIGHTS[fid]
        print(
            f"{fid:7s} | {flight.flight_type:8s} | T={p.track_id} | "
            f"FL={p.flight_level} | M={p.mach:.2f} | entry={p.entry_time:.0f} | "
            f"route={p.route_variant:7s} | wx={p.weather_exposure:7.1f} | "
            f"cost={p.total_cost:7.2f} | {describe_changes(flight, p)}"
        )

def solve_dynamic(config: ModelConfig = CONFIG) -> RestrictedSolution:
    # ------------------------------------------------------------
    # Stage 0: fast nominal screening. This performs only one
    # profile construction per flight and one pairwise conflict
    # check among the nominal trajectories. No MILP is created here.
    # ------------------------------------------------------------
    print("BUILDING NOMINAL TRAJECTORIES...")
    nominal_by_flight, infeasible_nominal = build_nominal_profiles(config)

    nominal_conflicts: List[ConflictInfo] = []
    if not infeasible_nominal:
        nominal_conflicts = selected_conflicts(nominal_by_flight, config)

    print("NOMINAL HARD-FEASIBILITY VIOLATIONS:", len(infeasible_nominal))
    print("NOMINAL CONFLICTS:", len(nominal_conflicts))

    if not infeasible_nominal and not nominal_conflicts:
        objective = sum(p.total_cost for p in nominal_by_flight.values())
        solution = RestrictedSolution(
            status="NominalConflictFree",
            objective=objective,
            selected_by_flight=nominal_by_flight,
            active_conflicts=[],
            conflict_constraints=0,
            paid_conflict_slacks=0,
        )
        print("NOMINAL TRAJECTORIES ARE CONFLICT-FREE. MILP WAS NOT BUILT.")
        print_final_solution(solution)
        return solution

    if infeasible_nominal:
        print(
            "DYNAMIC OPTIMIZATION REQUIRED BECAUSE NOMINAL PROFILES ARE "
            "HARD-INFEASIBLE FOR:",
            ", ".join(infeasible_nominal),
        )
    else:
        print("DYNAMIC OPTIMIZATION REQUIRED TO RESOLVE NOMINAL CONFLICTS.")

    # Exactly one seed profile per flight. Usually this is the nominal profile;
    # a small deterministic fallback is used only when nominal is infeasible.
    profile_pool: Dict[str, List[Profile]] = {
        fid: generate_initial_profiles(flight, config)
        for fid, flight in FLIGHTS.items()
    }

    history: List[Tuple[int, int, int, int, float | None]] = []
    last_solution: RestrictedSolution | None = None

    for iteration in range(config.max_dynamic_iterations + 1):
        total_profiles = sum(len(pool) for pool in profile_pool.values())
        print("\n" + "#" * 120)
        print(f"DYNAMIC ITERATION {iteration}")
        print("PROFILES BEFORE SOLVE:", total_profiles)

        solution = solve_restricted_master(profile_pool, config)
        last_solution = solution
        if solution.status not in {"Optimal", "Feasible"}:
            print("Restricted master failed with status:", solution.status)
            break

        n_conflicts = len(solution.active_conflicts)
        print("OBJECTIVE:", round(solution.objective or 0.0, 3))
        print("ACTIVE SELECTED CONFLICTS:", n_conflicts)
        print("PAID CONFLICT SLACKS:", solution.paid_conflict_slacks)
        print("CONFLICT CONSTRAINTS:", solution.conflict_constraints)

        if n_conflicts == 0:
            history.append((iteration, total_profiles, 0, 0, solution.objective))
            print("CONFLICT-FREE SOLUTION FOUND.")
            break

        conflicts_by_flight: Dict[str, List[ConflictInfo]] = defaultdict(list)
        for info in solution.active_conflicts:
            conflicts_by_flight[info.flight_a_id].append(info)
            conflicts_by_flight[info.flight_b_id].append(info)

        # Expand only a small repair set rather than both endpoints of every
        # conflict. This is a greedy weighted vertex-cover heuristic that
        # prioritizes requests over existing flights.
        repair_flights = choose_repair_flights(
            solution.active_conflicts, solution.selected_by_flight
        )
        print("REPAIR FLIGHTS SELECTED:", ", ".join(repair_flights))

        added = 0
        for fid in repair_flights:
            infos = conflicts_by_flight[fid]
            current = solution.selected_by_flight[fid]
            new_profiles = generate_targeted_profiles(
                flight=FLIGHTS[fid],
                current=current,
                conflict_infos=infos,
                selected_by_flight=solution.selected_by_flight,
                existing_pool=profile_pool[fid],
                config=config,
            )
            profile_pool[fid].extend(new_profiles)
            added += len(new_profiles)
            if new_profiles:
                print(
                    f"  {fid}: added {len(new_profiles)} targeted profiles; "
                    f"pool={len(profile_pool[fid])}"
                )

        history.append(
            (iteration, total_profiles, n_conflicts, added, solution.objective)
        )
        print("NEW PROFILES ADDED:", added)
        if added == 0:
            print("NO NEW FEASIBLE REPAIR PROFILES COULD BE GENERATED.")
            break

    if last_solution is None:
        raise RuntimeError("Dynamic solver did not run.")

    print("\nDYNAMIC HISTORY")
    print("iteration | profiles | conflicts | added | objective")
    for iteration, profiles, conflicts_n, added, objective in history:
        print(
            f"{iteration:9d} | {profiles:8d} | {conflicts_n:9d} | "
            f"{added:5d} | "
            f"{None if objective is None else round(objective, 3)}"
        )
    print_final_solution(last_solution)
    return last_solution


if __name__ == "__main__":
    solve_dynamic()

BUILDING NOMINAL TRAJECTORIES...
NOMINAL HARD-FEASIBILITY VIOLATIONS: 4
NOMINAL CONFLICTS: 0
DYNAMIC OPTIMIZATION REQUIRED BECAUSE NOMINAL PROFILES ARE HARD-INFEASIBLE FOR: EXG005, EXG006, REQ005, REQ006

########################################################################################################################
DYNAMIC ITERATION 0
PROFILES BEFORE SOLVE: 20
AVAILABLE SOLVERS: ['PULP_CBC_CMD']
OBJECTIVE: 100070.8
ACTIVE SELECTED CONFLICTS: 10
PAID CONFLICT SLACKS: 10
CONFLICT CONSTRAINTS: 10
REPAIR FLIGHTS SELECTED: REQ002, REQ009, REQ010, REQ006, REQ004, REQ008, REQ003, REQ007, REQ001, REQ005
  REQ002: added 4 targeted profiles; pool=5
  REQ009: added 4 targeted profiles; pool=5
  REQ010: added 4 targeted profiles; pool=5
  REQ006: added 4 targeted profiles; pool=5
  REQ004: added 4 targeted profiles; pool=5
  REQ008: added 4 targeted profiles; pool=5
  REQ003: added 4 targeted profiles; pool=5
  REQ007: added 4 targeted profiles; pool=5
  REQ001: added 4 targeted profiles;